# Actividad Integradora: Implementación Segura de AES-128 en Modo CBC
**Objetivo:** Corregir las vulnerabilidades de la implementación original mediante la correcta gestión del estado criptográfico (Llaves e IVs) y el empaquetado seguro de los datos.

## 1. Configuración del Entorno y Generación de Llaves
En esta sección importamos las librerías necesarias. 
La decisión arquitectónica clave aquí es abandonar las llaves predecibles ("hardcodeadas") y delegar la generación de entropía al sistema operativo utilizando un CSPRNG.

In [3]:
import os
import base64
from Crypto.Cipher import AES
from Crypto.Util.Padding import pad, unpad

# Requisito: Usar os.urandom(16) para la llave [cite: 49]
# Esto garantiza 128 bits reales de entropía criptográfica.
LLAVE_SEGURA = os.urandom(16)

MENSAJE = "Nermal, el lunes traeme lasagna extra porque Jon no sabe cocinar" 

print(f"[+] Llave maestra generada (hex): {LLAVE_SEGURA.hex()}")

[+] Llave maestra generada (hex): 3f64de5c2989bebcd46774f7752c099e


## 2. Lógica de Cifrado y Empaquetado
El diseño de la función de cifrado debe ser autosuficiente. El consumidor de esta función no debería tener que preocuparse por gestionar vectores de inicialización. 

**Flujo del algoritmo:**
1. Generar un IV efímero (único para esta sesión de cifrado).
2. Cifrar el payload.
3. Empaquetar el IV y el texto cifrado en una sola estructura de datos (concatenación).
4. Codificar la capa exterior en Base64 para garantizar su transporte seguro en sistemas de texto plano.

In [4]:
def cifrar_seguro(mensaje_texto, llave):
    
    
    iv = os.urandom(16) 
    
    cipher = AES.new(llave, AES.MODE_CBC, iv)
    
    
    mensaje_bytes = mensaje_texto.encode('utf-8')
    ct = cipher.encrypt(pad(mensaje_bytes, AES.block_size))
    
    
    paquete_completo = iv + ct
    
    return base64.b64encode(paquete_completo).decode('utf-8')

## 3. Lógica de Descifrado y Desempaquetado
La función de descifrado actúa como un pipeline inverso. Su arquitectura asume que el paquete entrante sigue el contrato de formato establecido por la función de cifrado (Base64 -> IV de 16 bytes + Ciphertext).

In [5]:
def descifrar_seguro(paquete_b64, llave):
    
    paquete_completo = base64.b64decode(paquete_b64)
    
    iv_extraido = paquete_completo[:16] 
    ct_extraido = paquete_completo[16:] 
    
    
    cipher = AES.new(llave, AES.MODE_CBC, iv_extraido)
    pt_padded = cipher.decrypt(ct_extraido)
    
    
    pt = unpad(pt_padded, AES.block_size)
    
    return pt.decode('utf-8')

## 4. Pruebas de Ejecución y Validación de Tamaños
Finalmente, instanciamos nuestro flujo para comprobar que la lógica funciona y verificamos la expansión de los datos, un fenómeno normal en criptografía debido al padding, el empaquetado del IV y el overhead que introduce la codificación en Base64 (aprox. 33% de aumento).